# 05_production_python — Production Python для ML Engineer

Практический ноутбук для подготовки к production-работе и интервью на ML Engineer.

Формат каждого раздела:
1. Теория
2. Код
3. Детальный разбор
4. Best practices
5. Вопросы собеседования
6. Мини-задачи


## 1) Logging best practices (`logging`)

### 1. Теория
Логирование — фундамент наблюдаемости. В production-ML логи нужны для расследования инцидентов, аудита пайплайнов, анализа ошибок модели и контроля SLA.

### 3. Детальный разбор
- Используйте централизованный `dictConfig`.
- Разделяйте уровни логов по смыслу: DEBUG/INFO/WARNING/ERROR/CRITICAL.
- Логируйте контекст: `batch_id`, `model_version`, `request_id`.

### 4. Best practices
- Не логируйте секреты и PII.
- Пишите структурированные поля, а не «чистый текст».
- Делайте лог-сообщения полезными для on-call инженера.

### 5. Вопросы собеседования
- Чем плох `print` в production?
- Когда использовать `logger.exception`?
- Почему параметризованные сообщения лучше f-string в logging?

### 6. Мини-задачи
- Настройте `RotatingFileHandler`.
- Добавьте correlation id.
- Разделите логгеры по модулям (`data`, `features`, `serving`).


In [ ]:
import logging
from logging.config import dictConfig

cfg = {
    "version": 1,
    "disable_existing_loggers": False,
    "formatters": {
        "default": {"format": "%(asctime)s | %(levelname)s | %(name)s | %(message)s"}
    },
    "handlers": {
        "console": {"class": "logging.StreamHandler", "formatter": "default", "level": "INFO"}
    },
    "root": {"handlers": ["console"], "level": "INFO"}
}

dictConfig(cfg)
logger = logging.getLogger("ml_service.inference")
logger.info("Старт инференса: model_version=%s", "v1.3.0")


## 2) Structured logging basics

### 1. Теория
Structured logging (JSON-подобные записи) упрощает фильтрацию, агрегацию и алертинг.

### 3. Детальный разбор
- Ключи должны быть стабильными (`event`, `service`, `env`, `trace_id`).
- Добавляйте технический и бизнес-контекст.

### 4. Best practices
- Используйте UTC timestamp.
- Контролируйте кардинальность полей.
- Храните схему событий логов.

### 5. Вопросы собеседования
- Почему структурированные логи лучше строковых?
- Какие поля обязательны для inference-логов?
- Что такое высокая кардинальность и чем она опасна?

### 6. Мини-задачи
- Сделайте JSON formatter.
- Добавьте маскирование чувствительных полей.
- Введите event taxonomy.


In [ ]:
import json
import logging
from datetime import datetime, timezone

class JsonFormatter(logging.Formatter):
    def format(self, record):
        payload = {
            "ts": datetime.now(timezone.utc).isoformat(),
            "level": record.levelname,
            "logger": record.name,
            "message": record.getMessage(),
            "event": getattr(record, "event", None),
            "model_version": getattr(record, "model_version", None),
        }
        return json.dumps(payload, ensure_ascii=False)


## 3) Exception handling patterns

### 1. Теория
Обработка исключений должна быть явной: либо восстановление, либо корректное падение с контекстом.

### 3. Детальный разбор
- Ловите конкретные исключения.
- Сохраняйте цепочку причины через `raise ... from ...`.
- Не «проглатывайте» ошибки.

### 4. Best practices
- Ретрай только для временных ошибок.
- На boundary-слое переводите ошибки в доменные коды/ответы.

### 5. Вопросы собеседования
- Почему `except Exception: pass` опасно?
- Что дает `raise ... from ...`?
- Какие ошибки в ML ETL можно ретраить?

### 6. Мини-задачи
- Добавьте backoff retry.
- Реализуйте fallback источник данных.
- Соберите карту исключений по слоям.


In [ ]:
def parse_positive_int(raw: str) -> int:
    try:
        value = int(raw)
    except ValueError as exc:
        raise ValueError(f"Некорректное число: {raw!r}") from exc
    if value <= 0:
        raise ValueError("Ожидается положительное число")
    return value


## 4) Custom exceptions

### 1. Теория
Custom exceptions описывают доменные ошибки и улучшают читаемость кода.

### 3. Детальный разбор
Иерархия исключений позволяет централизованно обрабатывать ошибки в API/worker.

### 4. Best practices
- Базовый класс доменных ошибок + специализированные подклассы.
- Единый стиль именования: `*Error`.

### 5. Вопросы собеседования
- Зачем нужен базовый `DomainError`?
- Как маппить custom exceptions в HTTP статусы?
- Почему слишком глубокая иерархия вредна?

### 6. Мини-задачи
- Добавьте `FeatureValidationError`.
- Добавьте `ModelNotReadyError`.
- Покройте тестами.


In [ ]:
class MLServiceError(Exception):
    pass

class DataValidationError(MLServiceError):
    pass

class ModelNotReadyError(MLServiceError):
    pass


## 5) Context managers (`__enter__`, `__exit__`)

### 1. Теория
Контекстные менеджеры гарантируют освобождение ресурсов даже при исключениях.

### 3. Детальный разбор
`__enter__` создает/захватывает ресурс, `__exit__` освобождает его.

### 4. Best practices
- Используйте `with` для файлов, коннектов, транзакций.
- Не подавляйте исключения без необходимости.

### 5. Вопросы собеседования
- Что значит вернуть `True` из `__exit__`?
- Почему `with` надежнее ручного `open/close`?
- Где в ML-пайплайне контекстные менеджеры критичны?

### 6. Мини-задачи
- Сделайте менеджер для временной директории.
- Добавьте логирование жизненного цикла ресурса.
- Сделайте менеджер таймера шага.


In [ ]:
class DBSession:
    def __enter__(self):
        print("open session")
        return self
    def __exit__(self, exc_type, exc, tb):
        print("close session")
        return False

with DBSession():
    print("query...")


## 6) Writing custom context managers

### 1. Теория
Кастомный менеджер полезен для измерения latency, временных настроек, трассировки.

### 3. Детальный разбор
`contextlib.contextmanager` удобен для лаконичной реализации.

### 4. Best practices
- В `finally` размещайте cleanup/метрики.
- Держите контекстные менеджеры узкими по ответственности.

### 5. Вопросы собеседования
- Когда использовать класс, а когда `@contextmanager`?
- Какие риски скрытого подавления исключений?
- Как стандартизировать измерение времени шагов?

### 6. Мини-задачи
- Добавьте отправку duration в метрики.
- Сделайте manager для временного уровня логирования.
- Сделайте manager для feature-flag override.


In [ ]:
import time
from contextlib import contextmanager

@contextmanager
def timed(name: str):
    t0 = time.perf_counter()
    try:
        yield
    finally:
        dt = (time.perf_counter() - t0) * 1000
        print(f"{name}: {dt:.2f} ms")


## 7) Typing and type hints

### 1. Теория
Аннотации типов делают контракты функций явными и упрощают командную разработку.

### 3. Детальный разбор
Типы особенно полезны на границах модулей: API, сервисы, репозитории, преобразование данных.

### 4. Best practices
- Типизируйте публичные функции в первую очередь.
- Используйте `TypedDict`, `Protocol`, `dataclass` по задаче.

### 5. Вопросы собеседования
- Какие runtime-ошибки типизация предотвращает заранее?
- Почему важны типы на границах слоев?
- Где typing в ML приносит максимальную пользу?

### 6. Мини-задачи
- Типизируйте pipeline preprocessing.
- Добавьте `TypedDict` для inference payload.
- Запустите mypy/pyright.


In [ ]:
from typing import Iterable

def normalize(xs: Iterable[float]) -> list[float]:
    vals = list(xs)
    mx = max(vals) if vals else 0.0
    return [x / mx for x in vals] if mx else [0.0 for _ in vals]


## 8) Static typing benefits

### 1. Теория
Статическая типизация уменьшает стоимость рефакторинга и снижает риск регрессий.

### 3. Детальный разбор
Type-checkers помогают обнаружить несовместимые интерфейсы до деплоя.

### 4. Best practices
- Включайте type-check в CI.
- Увеличивайте строгость правил постепенно.

### 5. Вопросы собеседования
- Почему typing не заменяет тесты?
- Какие ограничения у статической типизации в Python?
- Как внедрять typing в legacy-код?

### 6. Мини-задачи
- Включите strict режим для одного пакета.
- Добавьте `Protocol` для модельного интерфейса.
- Исправьте 10 предупреждений тайпчекера.


In [ ]:
from typing import Protocol

class Predictor(Protocol):
    def predict(self, features: list[float]) -> float:
        ...


## 9) Project structure best practices

### 1. Теория
Структура проекта должна поддерживать модульность, тестируемость и ясные границы ответственности.

### 3. Детальный разбор
`src-layout` предотвращает случайные импорты и улучшает reproducibility.

### 4. Best practices
- Отделяйте domain logic от infra.
- Выносите CLI/скрипты в отдельный каталог.
- Держите `tests/` зеркально к `src/`.

### 5. Вопросы собеседования
- Зачем нужен `src/` layout?
- Как разделить app/service/repository слои?
- Как организовать конфигурацию окружений?

### 6. Мини-задачи
- Спроектируйте структуру для online + batch.
- Добавьте модуль `common` для shared utilities.
- Создайте шаблон Makefile/CI.


In [ ]:
project/
├── pyproject.toml
├── src/ml_service/
├── tests/
└── scripts/


## 10) Virtual environments (`venv`, `pip`)

### 1. Теория
Изоляция зависимостей критична для воспроизводимости и стабильного деплоя.

### 3. Детальный разбор
`venv` изолирует зависимости проекта от системного Python.

### 4. Best practices
- Разделяйте runtime и dev зависимости.
- Храните lock-файлы.
- Не коммитьте `.venv`.

### 5. Вопросы собеседования
- Что ломается без изоляции окружения?
- Зачем lock-файлы?
- Как проверять воспроизводимость в CI?

### 6. Мини-задачи
- Настройте `requirements-dev.txt`.
- Добавьте bootstrap script.
- Проверьте сборку в чистом контейнере.


In [ ]:
# команды
# python -m venv .venv
# source .venv/bin/activate
# pip install -r requirements.txt


## 11) Packaging basics (`setup`, `pyproject`)

### 1. Теория
Пакетирование позволяет версионировать и переиспользовать код сервиса/библиотеки.

### 3. Детальный разбор
`pyproject.toml` — современный стандарт сборки Python-пакетов.

### 4. Best practices
- Следуйте SemVer.
- Делайте release notes.
- Автоматизируйте сборку в CI.

### 5. Вопросы собеседования
- Почему `pyproject.toml` лучше legacy-подхода?
- Что хранить в optional dependencies?
- Как связывать версии пакета и модели?

### 6. Мини-задачи
- Добавьте `dev` extra.
- Настройте editable install.
- Добавьте entrypoint для CLI.


In [ ]:
pyproject = {
    "project": {
        "name": "ml-service",
        "version": "0.1.0",
        "requires-python": ">=3.10",
    }
}


## 12) Testing with pytest

### 1. Теория
Тесты — страховка от регрессий. Для ML важны unit, integration и data-contract тесты.

### 3. Детальный разбор
Проверяйте граничные случаи, невалидные входы и стабильность API функций.

### 4. Best practices
- Используйте `parametrize`.
- Делайте изоляцию через фикстуры.
- Запускайте тесты в CI на каждом PR.

### 5. Вопросы собеседования
- Что обязательно тестировать в inference сервисе?
- Как избежать flaky tests?
- Где уместны интеграционные тесты?

### 6. Мини-задачи
- Добавьте фикстуру mock-model.
- Добавьте тест на custom exception.
- Введите coverage threshold.


In [ ]:
import pytest

def clip_prob(p: float) -> float:
    return 0.0 if p < 0 else 1.0 if p > 1 else p

@pytest.mark.parametrize("value, expected", [(-0.5, 0.0), (0.4, 0.4), (1.5, 1.0)])
def test_clip_prob(value: float, expected: float):
    assert clip_prob(value) == expected


## 13) Writing clean, maintainable code

### 1. Теория
Поддерживаемый код: маленькие функции, явные имена, минимум сайд-эффектов, проверяемые контракты.

### 3. Детальный разбор
Вынесение конфигурации в отдельные сущности снижает связность.

### 4. Best practices
- Принцип единственной ответственности.
- Явные границы модулей и зависимостей.
- Рефакторинг небольшими шагами + тесты.

### 5. Вопросы собеседования
- Что вы считаете признаком «грязного» кода?
- Как безопасно проводить рефакторинг в production?
- Почему понятные имена важнее «умных» сокращений?

### 6. Мини-задачи
- Разбейте длинную функцию на 3-4 шага.
- Добавьте валидацию входов.
- Добавьте докстринги публичному API.


In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Thresholds:
    fraud: float

def is_fraud(score: float, cfg: Thresholds) -> bool:
    if not 0 <= score <= 1:
        raise ValueError("score вне [0,1]")
    return score >= cfg.fraud


## 14) Typical ML project structure

### 1. Теория
Типовой production ML проект разделяет ingestion, features, training, serving и monitoring.

### 3. Детальный разбор
Раздельные контуры улучшают ownership, CI и эксплуатацию.

### 4. Best practices
- Исследовательские ноутбуки отдельно от production-кода.
- Версионируйте модели, признаки и схемы данных.
- Добавьте мониторинг качества модели и data drift.

### 5. Вопросы собеседования
- Где должна жить feature engineering логика?
- Как хранить артефакты модели?
- Какие SLO/SLA важны для inference сервиса?

### 6. Мини-задачи
- Спроектируйте структуру churn-проекта.
- Добавьте `contracts/` для схем.
- Опишите CI: lint -> typecheck -> test -> build -> deploy.


In [ ]:
ml_project/
├── configs/
├── src/ml_project/{data,features,training,serving,monitoring,common}
├── tests/{unit,integration}
└── notebooks/research


## Итог
Этот ноутбук покрывает ключевые production-компетенции Python для ML Engineer:
- наблюдаемость,
- надежность,
- типизацию,
- тестирование,
- архитектуру и упаковку проекта.
